# LAPORAN TUGAS BESAR — Klasifikasi Gangguan Sinyal XPQRS

**Mata Kuliah:** Kecerdasan Buatan  
**Topik:** Klasifikasi 17 Jenis Gangguan Sinyal Listrik dengan Deep Neural Network  
**Arsitektur:** Random Forest (baseline) + MLP + CNN 1D  
**Tanggal:** Juni 2026

> Notebook ini merupakan versi interaktif dari `LAPORAN_TUGAS_BESAR.md`.
> Laporan lengkap BAB 1–6 ada di file markdown tersebut.

---
# BAB 1 — PENDAHULUAN

## 1.1 Latar Belakang
Gangguan sinyal listrik (*sag*, *swell*, harmonisa, *flicker*, transien) dapat merusak peralatan dan mengganggu operasi industri. Dataset **XPQRS** menyediakan 17.000 sampel simulasi untuk mempelajari klasifikasi otomatis dengan DNN.

## 1.2 Rumusan Masalah
1. Bagaimana merancang pipeline deep learning end-to-end?
2. Preprocessing apa yang tepat tanpa data leakage?
3. Bagaimana perbandingan RF vs MLP vs CNN 1D?
4. Apa yang dipelajari model secara internal?

## 1.3 Tujuan & Batasan
- Pipeline lengkap: load → preprocess → train → evaluasi → visualisasi
- Dataset: XPQRS, 17 kelas, 100 timestep/sinyal
- Split: 80/10/10 stratified
- Framework: scikit-learn + PyTorch

---
# BAB 2 — TINJAUAN DATASET

| Parameter | Nilai |
|-----------|-------|
| Total sampel | 17.000 |
| Kelas | 17 (balanced, 1.000/kelas) |
| Panjang sinyal | 100 timestep (20 ms) |
| Sampling rate | 5 kHz |
| Format | `.mat` + 17 file `.csv` |

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from collections import Counter
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path('.')
MAT_PATH = ROOT / 'archive' / 'XPQRS' / '5Kfs_1Cycle_50f_1000Sam_1A.mat'
VIS_DIR = ROOT / 'visualizations'

CLASS_NAMES = [
    'Pure Sinusoidal', 'Sag', 'Swell', 'Interruption', 'Transient',
    'Oscillatory Transient', 'Harmonics', 'Harmonics with Sag',
    'Harmonics with Swell', 'Flicker', 'Flicker with Sag',
    'Flicker with Swell', 'Sag with Oscillatory Transient',
    'Swell with Oscillatory Transient', 'Sag with Harmonics',
    'Swell with Harmonics', 'Notch'
]

data = loadmat(str(MAT_PATH))
arr = data['Out']
X = arr.reshape((-1, arr.shape[1]))
y = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES)) for _ in range(arr.shape[0])]

print(f'Shape MAT: {arr.shape}  →  Flattened: {X.shape}')
print(f'Value range: [{X.min():.4f}, {X.max():.4f}]')
print(f'Balanced: {len(set(Counter(y).values())) == 1}')

In [ ]:
def show_viz(name, caption):
    path = VIS_DIR / name
    if path.exists():
        display(Markdown(f'**{caption}**'))
        display(Image(filename=str(path)))
    else:
        print(f'[Belum ada] {path} — jalankan: python -m src.visualize')

show_viz('waveforms.png', 'Gambar 2.1 — Bentuk gelombang beberapa kelas')
show_viz('signal_comparison.png', 'Gambar 2.2 — Domain waktu vs frekuensi')
show_viz('fft_sample.png', 'Gambar 2.3 — FFT Pure Sinusoidal')
show_viz('class_statistics.png', 'Gambar 2.4 — Statistik amplitudo per kelas')

---
# BAB 3 — METODOLOGI

## 3.1 Pipeline
```
Data → EDA → Split 80/10/10 → StandardScaler (fit train only) → Model → Evaluasi → Visualisasi
```

## 3.2 Preprocessing
- **Missing values:** tidak ditemukan; record invalid di-drop
- **StandardScaler:** fit hanya pada train (anti data leakage)
- **LabelEncoder:** 17 kelas → integer 0–16
- **CNN 1D reshape:** `(n, 100)` → `(n, 1, 100)`
- **Kelas seimbang:** tidak perlu oversampling

## 3.3 Split Data
| Set | Proporsi | Sampel |
|-----|----------|--------|
| Train | 80% | 13.600 |
| Val | 10% | 1.700 |
| Test | 10% | 1.700 |

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

encoder = LabelEncoder()
y_enc = encoder.fit_transform(y)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_enc, test_size=0.10, stratify=y_enc, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111111, stratify=y_temp, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(f'Train: {X_train_s.shape}  Val: {X_val_s.shape}  Test: {X_test_s.shape}')
print(f'Train scaled — mean: {X_train_s.mean():.6f}, std: {X_train_s.std():.6f}')

## 3.4 Arsitektur Model

### Random Forest (Baseline)
`RandomForestClassifier(n_estimators=100)` pada 9 fitur statistik per sinyal.

### MLP
```
Input(100) → Dense256+ReLU → Dense128+ReLU → Dense64+ReLU → Dense17+Softmax
Optimizer: Adam | Loss: Cross-Entropy | Batch: 128
```

### CNN 1D
```
Input(1,100) → Conv1d(32,k=3)+Pool → Conv1d(64,k=3)+Pool → Conv1d(128,k=3)+Pool
→ Flatten → Dense256+Dropout(0.5) → Dense128+Dropout(0.5) → Dense17 (logits)
Loss: CrossEntropyLoss | Optimizer: Adam lr=1e-3 | Batch: 128
```

## 3.5 Konfigurasi Training
- **Batch size 128:** kompromi stabilitas gradien vs efisiensi memori
- **Early stopping (CNN):** patience=10, best epoch=25
- **Regularisasi:** L2 (MLP), Dropout 0.5 (CNN)

---
# BAB 4 — HASIL DAN ANALISIS

## 4.1 Kurva Training

In [ ]:
show_viz('training_history_dnn.png', 'Gambar 4.1 — Kurva loss & akurasi MLP')
show_viz('training_history_cnn.png', 'Gambar 4.2 — Kurva loss & akurasi CNN 1D')

## 4.2 Tabel Performa (Test Set)

| Model | Input | Test Accuracy |
|-------|-------|:-------------:|
| Random Forest | 9 fitur statistik | **83,76%** |
| MLP | Raw signal (100) | 5,65% |
| CNN 1D | Raw signal (100) | 10,88% |
| Random guess | — | ~5,88% |

## 4.3 Confusion Matrix

In [ ]:
show_viz('dnn_confusion_matrix.png', 'Gambar 4.3 — Confusion Matrix MLP')
show_viz('cnn_confusion_matrix.png', 'Gambar 4.4 — Confusion Matrix CNN 1D')

In [ ]:
# Muat laporan training tersimpan
reports = {
    'Random Forest': ROOT / 'results' / 'training_rf_report.txt',
    'MLP (DNN)': ROOT / 'results' / 'training_dnn_report.txt',
    'CNN 1D': ROOT / 'results' / 'training_cnn_report.txt',
}

for name, path in reports.items():
    print('=' * 60)
    print(name)
    print('=' * 60)
    if path.exists():
        print(path.read_text(encoding='utf-8')[:1500])
    else:
        print(f'Belum ada: {path}')

---
# BAB 5 — PEMBAHASAN

## 5.1 Kesesuaian Arsitektur
- **CNN 1D** paling cocok secara teori untuk time-series (weight sharing, pola lokal).
- **MLP pada raw signal** kurang efektif — tiap timestep diperlakukan independen.
- **Random Forest** unggul karena fitur statistik sudah merangkum karakteristik domain.

## 5.2 Analisis Kurva Training
- MLP & CNN: val loss naik setelah beberapa epoch → **overfitting**.
- CNN sedikit lebih baik dari MLP (~11% vs ~6% test acc).
- Early stopping CNN di epoch 25 sudah tepat.

## 5.3 Keterbatasan
1. DNN akurasi masih rendah pada raw signal
2. 17 kelas dengan pola mirip (kombinasi gangguan)
3. Durasi sinyal pendek (20 ms)
4. Belum ada augmentasi data

---
# BAB 6 — KESIMPULAN DAN SARAN

## Kesimpulan
1. Pipeline end-to-end berhasil dibangun untuk 17 kelas gangguan sinyal XPQRS.
2. Preprocessing benar: StandardScaler setelah split, tanpa data leakage.
3. Random Forest (83,76%) >> CNN 1D (10,88%) > MLP (5,65%).
4. Visualisasi (waveform, FFT, kurva loss, confusion matrix) membantu interpretasi.
5. Hasil DNN kurang optimal namun **dapat dijelaskan secara teknis** — sesuai tujuan pembelajaran.

## Saran
1. Latih MLP dengan 28 fitur engineered (time + FFT)
2. Tambah data augmentation pada sinyal
3. Eksplorasi arsitektur CNN lebih dalam (BatchNorm, GAP)
4. Visualisasi bobot filter Conv1D untuk interpretabilitas
5. Rekam presentasi YouTube menjelaskan setiap keputusan teknis

---
# LAMPIRAN — Menjalankan Pipeline

```powershell
pip install -r requirements.txt
python -m src.load_data
python -m src.preprocess
python -m src.train_model
python -m src.train_dnn
python -m src.train_cnn --epochs 50 --batch-size 128
python -m src.visualize
```

Laporan lengkap: **`LAPORAN_TUGAS_BESAR.md`**